# Face Detection + Face Segmentation — Quick Demo

This notebook shows the 2-stage pipeline end-to-end on a single image. It assumes:

- `retinaface_best.pth` exists in `models/` (or pass `weights=None` to use random weights).
- `unet_best.pth` exists in `models/` (same caveat).
- You have at least one PNG/JPEG image you want to run on.

In [ ]:
import sys
from pathlib import Path

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from src.pipeline import FaceSegmentationPipeline

In [ ]:
# Build the pipeline. Pass `weights=None` if you don't have checkpoints yet.
pipe = FaceSegmentationPipeline(
    detector_weights='../models/retinaface_best.pth',
    segmentor_weights='../models/unet_best.pth',
    device='cpu',
    conf_threshold=0.7,
    nms_iou=0.5,
)

In [ ]:
# Run the pipeline on a single image.
IMAGE_PATH = '../data/processed/detection/test/0.jpg'  # <-- change to your file
result = pipe.run(IMAGE_PATH)
print(f"Detected {result.boxes.shape[0]} face(s). Failure reason: {result.failure_reason}")

In [ ]:
# Visualize the overlay (matplotlib expects RGB).
rgb = cv2.cvtColor(result.overlay, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 8))
plt.imshow(rgb)
plt.axis('off')
plt.title(f'Predictions ({result.boxes.shape[0]} faces)')
plt.show()

## Inspect individual masks

Each predicted mask is a single-channel uint8 array at the *crop* resolution.
The pipeline orchestrator pastes them back into the original image.

In [ ]:
for i, mask in enumerate(result.masks[:3]):  # show first 3
    plt.figure()
    plt.imshow(mask, cmap='gray')
    plt.title(f'Mask {i} (shape {mask.shape})')
    plt.axis('off')
    plt.show()